In [ ]:
import imageio as io
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
img_folder = "test_images"
stack = np.asarray(io.mimread(
    f"{img_folder}/out/2023-10-27_m157_yxz_100x_biocytin_ATTO647N_orig.tif"))
spines = np.asarray(io.mimread(
    f"{img_folder}/out/2023-10-27_m157_yxz_100x_biocytin_ATTO647N_spines.tif"))
dendrite = np.asarray(io.mimread(
    f"{img_folder}/out/2023-10-27_m157_yxz_100x_biocytin_ATTO647N_dendrite.tif"))

print(np.unique(dendrite))

In [ ]:
dendrite_mask = dendrite == 255


In [ ]:
plt.imshow(spines.max(0), cmap='gray')

In [ ]:
np.unique(spines)

In [ ]:
from skimage import measure
from skimage.measure import label, regionprops

In [ ]:
L = label(spines)

In [ ]:
plt.imshow(L.max(0), cmap='gray')
plt.colorbar()

In [ ]:
spine0 = L==1

In [ ]:
plt.imshow(spine0.max(0), cmap='gray')

In [ ]:
from skimage.measure import centroid

In [ ]:
cz, cy, cx = centroid(spine0)
plt.imshow(spine0.max(0), cmap='gray')  
plt.scatter(cx, cy, c='r', s=10)

In [ ]:
stack_masked = stack * spine0
plt.imshow(stack_masked.max(0), cmap='gray')
plt.xlim(1200,1400)
plt.ylim(420,320)


plt.scatter(cx, cy, c='r', s=10)


In [ ]:
def get_neighborhood(x, y, z, shape):
    x, y, z = int(round(x)), int(round(y)), int(round(z))
    neighbors = []
    for i in range(-1, 2):
        for j in range(-1, 2):
            for k in range(-1, 2):
                if i + j + k == 0:
                    continue
                nx, ny, nz = x + i, y + j, z + k
                if 0 <= nz < shape[0] and 0 <= ny < shape[1] and 0 <= nx < shape[2]:
                    neighbors.append((nx, ny, nz))
    return neighbors


In [ ]:
plt.imshow(stack_masked.max(0), cmap='gray')

for i in get_neighborhood(cx, cy, cz, spines.shape):
    print(i)
    plt.scatter(i[0], i[1], c=np.random.random(3), s=10)
    
plt.xlim(1200,1400)
plt.ylim(420,320)


In [ ]:
def floodfill(im, initial_mask):
    final_mask = np.zeros(im.shape, dtype=bool)
    final_mask[initial_mask] = True 

    cz, cy, cx = centroid(initial_mask)
    neighborhood = get_neighborhood(cx, cy, cz, im.shape)

    threshold = im[int(cz), int(cy), int(cx)] * 0.95

    while len(neighborhood) > 0:
        x, y, z = neighborhood.pop(0)
        if x < 0 or y < 0 or z < 0 or x >= im.shape[2] or y >= im.shape[1] or z >= im.shape[0]:
            continue
        if final_mask[z, y, x]:
            continue
        if im[z,y,x] > threshold:
            final_mask[z,y,x] = True
            neighborhood += get_neighborhood(x, y, z, im.shape)   
            
    return final_mask

In [ ]:
# i added
from collections import deque
import numpy as np

def floodfill_opt(im, initial_mask, max_voxels=None, forbidden_mask=None):
    """
    Flood-fills a spine region across Z slices from sparse mask annotations,
    using intensity and 3D connectivity. Optionally excludes 'forbidden' voxels.

    Parameters:
    - im: 3D image stack (Z, Y, X)
    - initial_mask: 3D binary mask for current spine
    - max_voxels: optional cap on max number of voxels to include
    - forbidden_mask: 3D binary mask where fill is not allowed (e.g., dendrites)

    Returns:
    - final_mask: 3D binary mask after flood fill
    """
    final_mask = np.zeros_like(im, dtype=bool)
    final_mask[initial_mask] = True

    alpha = 0.8

    # Use a high-intensity voxel from the mask as seed
    zyx_coords = np.argwhere(initial_mask)
    seed_z, seed_y, seed_x = zyx_coords[np.argmax(im[initial_mask])]
    seed_intensity = float(im[seed_z, seed_y, seed_x])
    threshold = min(seed_intensity * alpha, np.percentile(im[initial_mask], 85))

    # Initialize queue
    queue = deque()
    queue.extend(get_neighborhood(seed_x, seed_y, seed_z, im.shape))

    while queue:
        x, y, z = queue.popleft()

        # Bounds check
        if not (0 <= z < im.shape[0] and 0 <= y < im.shape[1] and 0 <= x < im.shape[2]):
            continue

        # Already visited or forbidden (like dendrite)
        if final_mask[z, y, x]:
            continue
        if forbidden_mask is not None and forbidden_mask[z, y, x]:
            continue

        # Intensity check
        if im[z, y, x] > threshold:
            final_mask[z, y, x] = True
            neighbors = get_neighborhood(x, y, z, im.shape)
            for nx, ny, nz in neighbors:
                if forbidden_mask is not None and forbidden_mask[nz, ny, nx]:
                    continue
                if final_mask[nz, ny, nx]:
                    continue
                queue.append((nx, ny, nz))


            if max_voxels is not None and np.count_nonzero(final_mask) > max_voxels:
                print("Flood fill aborted: too many voxels")
                break

    return final_mask


In [ ]:
from skimage.segmentation import find_boundaries

def floodfill_spine(img_stack, initial_mask, dendrite_mask, intensity_factor=0.8, max_voxels=None):
    """
    Flood-fill a spine region while avoiding dendrite areas.
    
    Parameters:
        img_stack: 3D image data (Z,Y,X)
        initial_mask: Binary mask of initial spine annotation
        dendrite_mask: Binary mask of dendrite (forbidden area)
        intensity_factor: Multiplier for intensity threshold (0.5-0.95)
        max_voxels: Optional limit to prevent overgrowth
        
    Returns:
        Binary mask of filled spine
    """
    # Initialize output mask
    filled_mask = np.zeros_like(initial_mask, dtype=bool)
    filled_mask[initial_mask] = True
    
    # Use the brightest voxel in initial mask as reference
    seed_z, seed_y, seed_x = np.unravel_index(
        np.argmax(img_stack * initial_mask),
        img_stack.shape
    )
    seed_intensity = img_stack[seed_z, seed_y, seed_x]
    threshold = seed_intensity * intensity_factor
    
    # Initialize queue with border voxels of initial mask
    border_voxels = find_boundaries(initial_mask, mode='inner')
    queue = list(zip(*np.where(border_voxels)))
    
    # Process queue
    while queue:
        z, y, x = queue.pop(0)
        
        # Check 6-connected neighborhood (more constrained than 26)
        for dz, dy, dx in [(0,0,1), (0,1,0), (1,0,0), 
                          (0,0,-1), (0,-1,0), (-1,0,0)]:
            nz, ny, nx = z+dz, y+dy, x+dx
            
            # Boundary check
            if not (0 <= nz < img_stack.shape[0] and 
                    0 <= ny < img_stack.shape[1] and 
                    0 <= nx < img_stack.shape[2]):
                continue
                
            # Skip if already filled or in dendrite
            if filled_mask[nz, ny, nx] or dendrite_mask[nz, ny, nx]:
                continue
                
            # Intensity check
            if img_stack[nz, ny, nx] > threshold:
                filled_mask[nz, ny, nx] = True
                queue.append((nz, ny, nx))
                
                # Optional early stopping
                if max_voxels and np.sum(filled_mask) > max_voxels:
                    return filled_mask
                    
    return filled_mask

In [ ]:
def floodfill_spine_26(img_stack, initial_mask, dendrite_mask, intensity_factor=0.7, z_penalty=1.2):
    filled_mask = np.zeros_like(initial_mask)
    queue = deque()
    
    # Seed from brightest voxel in annotation
    seed = np.unravel_index(np.argmax(img_stack * initial_mask), img_stack.shape)
    queue.append(seed)
    filled_mask[seed] = True
    
    ref_intensity = img_stack[seed]
    
    # 26-neighbor offsets
    neighbors = [(dz,dy,dx) for dz in (-1,0,1) 
                          for dy in (-1,0,1) 
                          for dx in (-1,0,1) 
                          if (dz,dy,dx) != (0,0,0)]
    
    while queue:
        z,y,x = queue.popleft()
        
        for dz, dy, dx in neighbors:
            nz, ny, nx = z+dz, y+dy, x+dx
            
            # Boundary checks
            if (nz < 0 or nz >= img_stack.shape[0] or 
                ny < 0 or ny >= img_stack.shape[1] or 
                nx < 0 or nx >= img_stack.shape[2]):
                continue
                
            # Skip if already filled or in dendrite
            if filled_mask[nz,ny,nx] or dendrite_mask[nz,ny,nx]:
                continue
            
            # Z-axis penalty for inter-plane connections
            current_threshold = ref_intensity * intensity_factor
            if dz != 0:  # If crossing Z-planes
                current_threshold *= z_penalty  # Require brighter voxels
                
            if img_stack[nz,ny,nx] > current_threshold:
                filled_mask[nz,ny,nx] = True
                queue.append((nz,ny,nx))
    
    return filled_mask

In [ ]:
final_mask = floodfill(stack, L==10)

io.mimwrite(f"{img_folder}/out/2023-10-27_m157_yxz_100x_biocytin_ATTO647N_flood.tif", final_mask.astype(np.uint8) * 255)
plt.imshow(final_mask.max(0), cmap='gray')

In [ ]:
diff = (final_mask.astype(int) - (L == 21).astype(int)).astype(np.int8)

plt.imshow(diff.max(0), cmap='bwr')  # Blue = lost, Red = added
plt.title("Flood fill difference (red = added, blue = lost)")
plt.colorbar()

In [ ]:
# looping thru all the labels -------------------------------------------

from skimage.measure import label
import numpy as np
import matplotlib.pyplot as plt

# Get all spine labels (exclude background = 0)
labels = np.unique(L)
labels = labels[labels != 0]

# Final mask to accumulate filled results
final_mask = np.zeros_like(L, dtype=bool)

# Optional: to store diffs if you want to analyze later
diff_map = np.zeros_like(L, dtype=np.int8)

# Extend dendrite mask over the whole Z axis
dendrite_mask_combined = np.any(dendrite_mask, axis=0)
dendrite_mask_broad = np.broadcast_to(dendrite_mask_combined, dendrite_mask.shape)

# Loop through all spines
for label_id in labels:
    mask = L == label_id
    filled = floodfill_opt(stack, mask, forbidden_mask=dendrite_mask_broad)

    final_mask |= filled  # accumulate into final mask

    # Optional: difference map
    diff = filled.astype(int) - mask.astype(int)
    diff_map += diff.astype(np.int8)  # accumulate all diffs

    # Optional: Per-spine difference plot
    # plt.figure()
    # plt.imshow(diff.max(0), cmap='bwr', vmin=-1, vmax=1)
    # plt.title(f"Spine {label_id} - red = added, blue = lost")
    # plt.colorbar()
    # plt.show()

# Save or visualize final mask
plt.figure()
plt.imshow(final_mask.max(0), cmap='gray')
plt.title("Final Accumulated Mask (Max Projection)")
plt.axis('off')
plt.show()

# Show total diff (accumulated)
plt.figure()
plt.imshow(diff_map.max(0), cmap='bwr', vmin=-1, vmax=1)
plt.title("Total Difference Map: red = added, blue = lost")
plt.colorbar()
plt.axis('off')
plt.show()


In [ ]:
# Process all spines
final_mask = np.zeros_like(stack, dtype=bool)

for spine_id in np.unique(L)[1:]:  # Skip background
    spine_mask = (L == spine_id)
    filled_spine = floodfill_spine_26(
        stack, 
        spine_mask,
        dendrite_mask_broad,
        # dendrite_mask,
        intensity_factor=0.7,  # Adjust based on your data
        z_penalty = 1.2
    )
    final_mask = final_mask | filled_spine | spine_mask

# Visualize
plt.figure(figsize=(12,6))
plt.subplot(121)
plt.imshow(final_mask.max(0), cmap='gray')
plt.title('Flood-filled spines')
plt.subplot(122)
plt.imshow(dendrite_mask.max(0), cmap='gray')
plt.title('Dendrite mask')
plt.show()

In [ ]:
# different masks difference ---------------------------------------------
label_id = 1
mask = L == label_id
filled = floodfill(stack, mask)
filled2 = floodfill_opt(stack, mask)

# Compute max projection along Z
mask_proj     = mask.max(axis=0)
filled_proj   = filled.max(axis=0)
filled2_proj  = filled2.max(axis=0)

# Difference maps
diff1_proj = (filled.astype(int) - mask.astype(int)).max(axis=0)
diff2_proj = (filled2.astype(int) - mask.astype(int)).max(axis=0)

# Plot
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

axes[0].imshow(mask_proj, cmap='gray')
axes[0].set_title("Original Mask")

axes[1].imshow(filled_proj, cmap='gray')
axes[1].set_title("Flood Fill")

axes[2].imshow(filled2_proj, cmap='gray')
axes[2].set_title("Flood Fill (Optimized)")

axes[3].imshow(diff1_proj, cmap='bwr', vmin=-1, vmax=1)
axes[3].set_title("Diff: Fill vs Mask")

axes[4].imshow(diff2_proj, cmap='bwr', vmin=-1, vmax=1)
axes[4].set_title("Diff: Optimized vs Mask")

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Original mask size:", mask.sum())
print("Flood-filled size:", filled.sum())
print("Flood-filled optimized size:", filled2.sum())
print("Added voxels for first version:", filled.sum() - mask.sum())
print("Added voxels for second version:", filled2.sum() - mask.sum())


In [ ]:
# checking spines one by one and overlaying --------------------------------------------------
for label_id in labels[:10]:  # or loop over all if fast enough
    mask = L == label_id

    filled_v1 = floodfill(stack, mask)
    filled_v2 = floodfill_opt(stack, mask, forbidden_mask=dendrite_mask_broad)

    # Compute overlays
    only_original = mask & ~filled_v1 & ~filled_v2
    only_v1 = filled_v1 & ~mask & ~filled_v2
    only_v2 = filled_v2 & ~mask & ~filled_v1
    v1_and_v2 = filled_v1 & filled_v2 & ~mask
    all_three = filled_v1 & filled_v2 & mask

    # Create RGB image to overlay
    overlay = np.zeros((*mask.shape[1:], 3), dtype=np.uint8)

    overlay[only_original.max(0)] = [255, 255, 255]  # white = original
    overlay[only_v1.max(0)] = [0, 255, 0]            # green = added by v1
    overlay[only_v2.max(0)] = [0, 0, 255]            # blue = added by v2
    overlay[v1_and_v2.max(0)] = [255, 255, 0]        # yellow = added by both
    overlay[all_three.max(0)] = [255, 0, 0]          # red = overlapping all

    plt.figure(figsize=(5, 5))
    plt.imshow(overlay)
    plt.title(f"Spine {label_id} Comparison\nWhite=Original | Green=v1 | Blue=v2 | Yellow=Both Added | Red=All 3")
    plt.axis('off')
    plt.show()
